# Supervised Fine-Tuning

**Supervised Fine-Tuning (SFT)** is mainly used to adapt pre-trained language models so they follow instructions, hold a dialogue, and use specific output formats. Pre-trained models have impressive general capabilities, and SFT turns them into assistant-like models that understand and respond to user prompts better. This is usually done by training on datasets of human-written conversations and instructions.

This page is a step-by-step guide to fine-tuning the `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` model with the `SFTTrainer`. Following these steps, you can adapt the model to perform specific tasks more effectively.

---

## When to Use SFT

Before getting into the implementation, it's important to know when SFT is the right choice for your project. First, consider whether an existing instruction-tuned model with well-crafted prompts would be enough for your use case. SFT takes significant computing resources and engineering effort, so only pursue it when prompting existing models isn't enough.

Consider SFT only if you:

- Need more performance than prompting can achieve
- Have a specific use case where the cost of using a large general-purpose model outweighs the cost of fine-tuning a smaller model
- Require specialized output formats or domain-specific knowledge that existing models struggle with

If you decide SFT is necessary, whether to go ahead depends on two main factors.

### Template Control

SFT gives you precise control over the structure of the model's output. This is especially valuable when you need the model to:

1. Generate responses in a specific chat template format
2. Follow strict output schemas
3. Keep a consistent style across responses

### Domain Adaptation

When you work in a specialized domain, SFT helps align the model with that domain's requirements by:

1. Teaching domain terminology and concepts
2. Enforcing professional standards
3. Handling technical queries appropriately
4. Following industry-specific guidelines

---

## Before You Start

Before starting SFT, check whether your use case requires:

- [ ] Precise output formatting
- [ ] Domain-specific knowledge
- [ ] Consistent response patterns
- [ ] Adherence to specific guidelines

This check will help you decide whether SFT is the right approach for your needs.

## Training Configuration

How well your fine-tuning works depends heavily on choosing the right training parameters. The `SFTTrainer` configuration has several parameters that control the training process. Here's what each important one does and how to set it.

### 1. Training Duration Parameters

| Parameter | Purpose |
|---|---|
| `num_train_epochs` | Controls the total training duration (how many passes over the dataset) |
| `max_steps` | An alternative to epochs that sets the maximum number of training steps |

More epochs let the model learn more, but they risk **overfitting**.

### 2. Batch Size Parameters

| Parameter | Purpose |
|---|---|
| `per_device_train_batch_size` | Determines memory usage and training stability |
| `gradient_accumulation_steps` | Enables larger effective batch sizes |

Larger batches give more stable gradients but need more memory.

> **Effective batch size** = `per_device_train_batch_size` × `gradient_accumulation_steps` × number of GPUs

### 3. Learning Rate Parameters

| Parameter | Purpose |
|---|---|
| `learning_rate` | Controls the size of each weight update |
| `warmup_ratio` | The portion of training used for learning rate warmup |

A learning rate that's **too high** can make training unstable, and one that's **too low** makes learning slow.

### 4. Monitoring Parameters

| Parameter | Purpose |
|---|---|
| `logging_steps` | How often metrics are logged |
| `eval_steps` | How often the model is evaluated on validation data |
| `save_steps` | How often a model checkpoint is saved |

---

### Recommended Starting Point

Start with conservative values and adjust based on what monitoring shows:

- Begin with **1–3 epochs**
- Use **smaller batch sizes** at first
- **Monitor validation metrics** closely
- **Adjust the learning rate** if training is unstable

### Example Configuration

```python
from trl import SFTConfig

config = SFTConfig(
    output_dir="./sft-output",

    # Training duration
    num_train_epochs=3,                 # or use max_steps instead

    # Batch size
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size = 16

    # Learning rate
    learning_rate=5e-5,
    warmup_ratio=0.1,                   # first 10% of steps warm up

    # Monitoring
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
)
```

In [4]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, clone_chat_template
import torch

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = "HuggingFaceTB/SmolLM2-135M"

In [6]:
dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

Generating test split: 100%|██████████| 54948/54948 [00:00<00:00, 108086.95 examples/s]


In [7]:
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=checkpoint).to(device)

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\models--HuggingFaceTB--SmolLM2-135M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1168.56it/s]


In [8]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=checkpoint)

In [9]:
model, tokenizer, added_tokens = clone_chat_template(model, tokenizer, "HuggingFaceTB/SmolLM2-135M-Instruct")

In [10]:
training_args = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy='steps',
    eval_steps=50
)
"""
 To enable packing, simply set 'packing=True' in the SFTConfig constructor.
 This feature allows multiple short examples to be packed into the same input sequence, maximizing GPU utilization during training.
"""

In [11]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

Truncating train dataset: 100%|██████████| 1043917/1043917 [03:52<00:00, 4481.58 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 1043917/1043917 [00:58<00:00, 17765.95 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 54948/54948 [00:03<00:00, 17170.10 examples/s]


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
